# Get one metadata table for the different results

EGA_meta is the computationally determined metadata and Final_meta corresponds to the data manually collated. We double checked they had the same values, and ega_met is only pulled in here to 1) show how many samples we lose by not having their clinical data or RNA-seq data, and 2) to get the Source info.

In [22]:
library(data.table)
library(readxl)
library(dplyr)

In [23]:
ega_meta <- fread("./EGA_rel_data/Harmonized_EGA_pat_data.txt", fill=TRUE, header=TRUE, sep="\t") # Made from Harmonizing_Metadata.ipynb
rest_meta <- fread("./Harmonization/NonEGA_metadata_8.19.25.txt", fill=TRUE, sep="\t") # Made from Get_metadata.ipynb in Nanostring/analyses

final_meta <- read_excel("./EGA_rel_data/Final_Samples.xlsx")

colnames(ega_meta)
colnames(rest_meta)
colnames(final_meta)


[1] "Patient"                              
 [2] "Data_Found_In"                        
 [3] "PlatFI_12_months"                     
 [4] "PlatFI_days"                          
 [5] "FIGO_Stage"                           
 [6] "Tissues"                              
 [7] "Sample_stages"                        
 [8] "Response_to_Treatment"                
 [9] "Age"                                  
[10] "OvaHRDscar"                           
[11] "BRCAmut_status"                       
[12] "CRS"                                  
[13] "CA125_Uperml_TN_PN"                   
[14] "Piet21_Survival"                      
[15] "Piet21Residual_tumor_after_DS"        
[16] "Piet21_Primary_chemo_regimen"         
[17] "Piet21_Progression_free_survival_mths"
[18] "Piet21_Overall_survival_mths"         
[19] "Piet21_Relapse_type"

[1] "Pre_count_name"                   "Post_count_name"                 
 [3] "Source"                           "Age"                             
 [5] "PFS"                              "Recurrence"                      
 [7] "PFS_high"                         "Chemo_status"                    
 [9] "PFS_high_3298797"                 "OS"                              
[11] "Death"                            "Patient"                         
[13] "Name_Source"                      "Stage"                           
[15] "PFI_mth"                          "james_progression_status_4.25.22"
[17] "manso_response"                   "CRS"                             
[19] "PFS_high_32928797"                "Chemo_regimen"

[1] "Patient"            "Final_Bool"         "Assay"             
 [4] "RNAseqID"           "Twin"               "OG_Tissue"         
 [7] "Twin_Tissue"        "Stage"              "Death"             
[10] "Res_Tumor_afterPDS" "Chemo_regimen"      "Therapy_outcome"   
[13] "PFS_mth"            "PFI_days"           "OS_mths"           
[16] "PFI_daysZhang"      "CRS"                "CA125_Uperml_TN"   
[19] "CA125_Uperml_PN"    "Age_Zhang"          "Final_PFI_days"    
[22] "PFI_source"

In [24]:
# convert PFI_days to months?
rest_meta$PFI_days <- as.numeric(rest_meta$PFI_mth)*30
# convert pFI_mth to days
final_meta$PFI_mth <- as.numeric(final_meta$Final_PFI_days)/30


In [25]:
print("OS")
quantile(rest_meta$OS, na.rm=TRUE)
quantile(as.numeric(final_meta$OS_mths), na.rm=TRUE)
print("PFS")
quantile(rest_meta$PFS, na.rm=TRUE)
quantile(as.numeric(final_meta$PFS_mth), na.rm=TRUE)
print("PFI months")
quantile(rest_meta$PFI_mth, na.rm=TRUE)
quantile(as.numeric(final_meta$PFI_mth), na.rm=TRUE)

print("PFI days")
quantile(rest_meta$PFI_days, na.rm=TRUE)
quantile(as.numeric(final_meta$Final_PFI_days), na.rm=TRUE)

print("Chemo status and regimen")
unique(rest_meta$Chemo_status)
unique(final_meta$Chemo_regimen)

unique(rest_meta$manso_response)

[1] "OS"


0%       25%       50%       75%      100% 
  2.10000  15.96667  29.80000  47.00000 106.00000

Warning message in quantile(as.numeric(final_meta$OS_mths), na.rm = TRUE):
“NAs introduced by coercion”


0%   25%   50%   75%  100% 
 9.00 18.75 30.00 38.45 54.00

[1] "PFS"


...

0%  25%  50%  75% 100% 
   0   36  203  511 1137

[1] "Chemo status and regimen"


[1]  1  2  0 NA

[1] "CP-TX-BV" NA         "CP-TX"    "CP"

[1] NA              "Responder"     "Non-responder"

In [26]:
colnames(rest_meta)

[1] "Pre_count_name"                   "Post_count_name"                 
 [3] "Source"                           "Age"                             
 [5] "PFS"                              "Recurrence"                      
 [7] "PFS_high"                         "Chemo_status"                    
 [9] "PFS_high_3298797"                 "OS"                              
[11] "Death"                            "Patient"                         
[13] "Name_Source"                      "Stage"                           
[15] "PFI_mth"                          "james_progression_status_4.25.22"
[17] "manso_response"                   "CRS"                             
[19] "PFS_high_32928797"                "Chemo_regimen"                   
[21] "PFI_days"

### Just get patient info for now

In [27]:
rest_meta$Res_Tumor_afterPDS <- NA
rest_meta$Chemo_regimen <- "NA"
rest_meta$Therapy_outcome <- "NA"
rest_meta$CA125_Uperml_TN <- NA
rest_meta$CA125_Uperml_PN <- NA


rest_patient_meta <- rest_meta[, c("Patient", "Death", "Stage", "Name_Source", "Age", "PFS", "PFI_mth", "PFI_days", "OS", "CRS",
                                   "Recurrence", "Therapy_outcome", "Chemo_status", "Chemo_regimen", "PFS_high", "PFS_high_32928797", "manso_response", 
                                   "james_progression_status_4.25.22", "Res_Tumor_afterPDS", "CA125_Uperml_TN", "CA125_Uperml_PN")]
final_meta$Recurrence <- "NA"
final_meta$PFS_high <- "NA"
final_meta$Chemo_status <- "NA"
final_meta$PFS_high_32928797 <- "NA"
final_meta$Manso_Response <- "NA"
final_meta["James_Prog_status_4.25.22"] <- "NA"

final_patient_meta <- final_meta[, c("Patient", "Death", "Stage", "PFI_source", "Age_Zhang", "PFS_mth", "PFI_mth", "Final_PFI_days", "OS_mths", "CRS", 
                                     "Recurrence", "Therapy_outcome", "Chemo_status", "Chemo_regimen", "PFS_high", "PFS_high_32928797", "Manso_Response", 
                                     "James_Prog_status_4.25.22", "Res_Tumor_afterPDS", "CA125_Uperml_TN", "CA125_Uperml_PN")]

comb_colnames <- c("Patient", "Death", "Stage", "Source", "Age", "PFS_mths", "PFI_mths", "PFI_days", "OS_mths", "CRS", 
                   "Recurrence", "Therapy_outcome", "Chemo_status", "Chemo_regimen", "PFS_high", "PFS_high_32928797", 
                   "Manso_Response", "James_Prog_status_4.25.22", "Res_Tumor_afterPDS", "CA125_Uperml_TN", "CA125_Uperml_PN")
colnames(rest_patient_meta) <- comb_colnames
colnames(final_patient_meta) <- comb_colnames

comb_patient_meta <- rbind(rest_patient_meta, final_patient_meta)

comb_patient_meta <- comb_patient_meta %>%
  filter(!if_all(-c(Patient,Source), ~ is.na(.) | . == "NA" | . == ""))  %>% # keep rows where not all (except first col) are NA
  distinct() 
dim(comb_patient_meta)

[1] 163  21

In [28]:
rest_patient_meta[1:4,]

Patient,Death,Stage,Source,Age,PFS_mths,PFI_mths,PFI_days,OS_mths,CRS,⋯,Therapy_outcome,Chemo_status,Chemo_regimen,PFS_high,PFS_high_32928797,Manso_Response,James_Prog_status_4.25.22,Res_Tumor_afterPDS,CA125_Uperml_TN,CA125_Uperml_PN
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>,<int>,⋯,<chr>,<int>,<chr>,<int>,<lgl>,<chr>,<chr>,<lgl>,<lgl>,<lgl>
P1,1,NA,Bitler,55,8,NA,NA,44,NA,⋯,NA,1,NA,0,NA,NA,NA,NA,NA,NA
P2,0,NA,Bitler,54,13,NA,NA,51,NA,⋯,NA,2,NA,1,NA,NA,NA,NA,NA,NA
P3,1,NA,Bitler,56,3,NA,NA,35,NA,⋯,NA,0,NA,0,NA,NA,NA,NA,NA,NA
P4,1,NA,Bitler,48,9,NA,NA,27,NA,⋯,NA,1,NA,0,NA,NA,NA,NA,NA,NA


In [29]:
dim(comb_patient_meta)
length(unique(comb_patient_meta$Patient))
comb_patient_meta$Patient[duplicated(comb_patient_meta$Patient)]
table(comb_patient_meta$Source)

comb_patient_meta[comb_patient_meta$Patient == "WSU_F",]

comb_patient_meta$uniq_patient_name <- paste0(comb_patient_meta$Patient, "_", comb_patient_meta$Source)
length(unique(comb_patient_meta$uniq_patient_name))

[1] 163  21

[1] 115

[1] "P1"  "P2"  "P3"  "P4"  "P5"  "P6"  "P7"  "P8"  "P9"  "P10" "P11" "P12"
[13] "P13" "P14" "P15" "P16" "P17" "P18" "P19" "P20" "P21" "P22" "P23" "P24"
[25] "P25" "P26" "P27" "P28" "P29" "P30" "P31" "P1"  "P2"  "P17" "P3"  "P4" 
[37] "P5"  "P6"  "P7"  "P8"  "P9"  "P10" "P11" "P12" "P13" "P14" "P15" "P16"


Adzib_2023     Bitler      James    Jav_RNA      Manso       Piet      Zhang 
        15         35         31         36         17          7          3 
Zhang_Piet 
        19 

Patient,Death,Stage,Source,Age,PFS_mths,PFI_mths,PFI_days,OS_mths,CRS,⋯,Therapy_outcome,Chemo_status,Chemo_regimen,PFS_high,PFS_high_32928797,Manso_Response,James_Prog_status_4.25.22,Res_Tumor_afterPDS,CA125_Uperml_TN,CA125_Uperml_PN
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
WSU_F,NA,NA,Adzib_2023,55,18,NA,NA,43,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


[1] 163

In [30]:
write.table(comb_patient_meta, "./Harmonization/FullPatient_metadata_9.16.25.txt", quote=FALSE, 
           sep="\t", row.names=FALSE)

### Check how many we have if we consider different data for response

In [31]:
cat("\nNumber of patients with non-NA PFS_mths WITH the important caveat that James PFS are actually PFIs.")
dim(comb_patient_meta[!is.na(comb_patient_meta$PFS_mths),])
table(comb_patient_meta[!is.na(comb_patient_meta$PFS_mths),]$Source)


cat("\nNumber of patients with non-NA PFI_mths:")
dim(comb_patient_meta[!is.na(comb_patient_meta$PFI_mths),])
table(comb_patient_meta[!is.na(comb_patient_meta$PFI_mths),]$Source)
cat('\t\t So we lose all Manso Nano (17) and Adzib RNA (15)')


Number of patients with non-NA PFS_mths WITH the important caveat that James PFS are actually PFIs.

[1] 159  22


Adzib_2023     Bitler      James    Jav_RNA      Manso       Piet Zhang_Piet 
        15         35         31         36         17          7         18 


Number of patients with non-NA PFI_mths:

[1] 60 22


     James       Piet      Zhang Zhang_Piet 
        31          7          3         19 

		 So we lose all Manso Nano (17) and Adzib RNA (15)